# Week 2 – Day 4: Imbalanced Classification & Performance Evaluation

## Objective & Scope
This notebook systematically tackles class imbalance in binary classification tasks (e.g., fraud detection, anomaly triage, churn prediction):
1. **The Accuracy Paradox:** Demonstrating how baseline majority classifiers exploit severe class imbalance (95% vs 5%) to report inflated accuracy while completely failing at minority detection.
2. **Cost-Sensitive Learning:** Evaluating the impact of loss re-weighting (`class_weight='balanced'` and `scale_pos_weight`) across Logistic Regression, Random Forest, and XGBoost.
3. **Discriminative Evaluation:** Comparing Receiver Operating Characteristic (ROC) and Precision-Recall (PR) curves, proving why PR curves are essential when negative instances dominate.
4. **Optimal Decision Thresholding:** Implementing a business-constrained threshold optimizer to guarantee target recall while maximizing precision.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score
)

# 1. Synthesize heavily imbalanced binary classification dataset (95% Negatives, 5% Positives)
X, y = make_classification(
    n_samples=3000,
    n_features=20,
    n_informative=10,
    n_redundant=4,
    weights=[0.95, 0.05],
    flip_y=0.01,
    random_state=42
)

# Stratified 80/20 train/test split preserving the exact 95:5 class ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f"Total Samples: {len(X)} | Training: {len(X_train)} | Testing: {len(X_test)}")
print(f"Training Class Distribution: Negative={np.sum(y_train==0)}, Positive={np.sum(y_train==1)} ({np.mean(y_train)*100:.1f}%)")
print(f"Test Class Distribution    : Negative={np.sum(y_test==0)}, Positive={np.sum(y_test==1)} ({np.mean(y_test)*100:.1f}%)")

In [ ]:
# --- TASK 1: THE MAJORITY CLASS ACCURACY PARADOX ---
majority_baseline = DummyClassifier(strategy='most_frequent')
majority_baseline.fit(X_train, y_train)
y_baseline_pred = majority_baseline.predict(X_test)

print("=== Majority-Class Baseline Performance ===")
print(f"Accuracy : {accuracy_score(y_test, y_baseline_pred):.4f} (Deceptively high!)")
print(f"Precision: {precision_score(y_test, y_baseline_pred, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_test, y_baseline_pred):.4f} (Zero detections of minority event)")
print(f"F1-Score : {f1_score(y_test, y_baseline_pred):.4f}")

In [ ]:
# --- TASK 1: DEFAULT VS COST-SENSITIVE LOGISTIC REGRESSION ---
# 1. Default Logistic Regression
lr_unweighted = LogisticRegression(max_iter=1000, random_state=42).fit(X_train, y_train)
y_pred_unweighted = lr_unweighted.predict(X_test)

print("=== Standard (Unweighted) Logistic Regression ===")
print(f"Recall: {recall_score(y_test, y_pred_unweighted):.4f} | F1: {f1_score(y_test, y_pred_unweighted):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_unweighted))

# 2. Balanced Logistic Regression
lr_balanced = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42).fit(X_train, y_train)
y_pred_balanced = lr_balanced.predict(X_test)

print("\n=== Balanced (Cost-Sensitive) Logistic Regression ===")
print(f"Recall: {recall_score(y_test, y_pred_balanced):.4f} | F1: {f1_score(y_test, y_pred_balanced):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_balanced))
print("\nDetailed Classification Report:\n", classification_report(y_test, y_pred_balanced))

### Task 1 & 2: Evaluation Metrics & Architectural Rationale

#### 1. Confusion Matrix Breakdown in a Financial Fraud Detection Context
* **True Positive (TP):** A fraudulent credit card charge is correctly flagged and blocked by the model, preventing immediate financial loss.
* **False Positive (FP):** A legitimate customer purchase is mistakenly blocked as suspicious, creating friction and customer annoyance.
* **False Negative (FN):** An actual fraudulent transaction slips through unnoticed, causing direct economic damage, liability disputes, and reputational loss.
* **True Negative (TN):** A legitimate transaction is correctly cleared without interruption.

#### 2. Mechanistic Effect of `class_weight='balanced'`
Under default cross-entropy loss, each training row contributes equally. Because 95% of rows are non-fraud, the gradient descent steps are dominated by minimizing error on class 0. By setting `class_weight='balanced'`, the loss function weights errors inversely proportional to class frequencies:
$$\text{weight}(c) = \frac{N}{2 \times N_c}$$
For a 95:5 imbalance, minority errors are penalized $\sim 19\times$ more severely, forcing the optimization algorithm to shift the decision boundary to prioritize minority recall.

#### 3. When to Use ROC-AUC vs. F1-Score
1. **Use ROC-AUC during initial model benchmarking:** ROC-AUC evaluates ranking quality across all possible operating thresholds ($0.0$ to $1.0$). It tells you whether the model consistently scores positive cases higher than negative cases, independent of any arbitrary cutoff.
2. **Use F1-Score during business deployment:** Once a concrete operating threshold is selected, F1-Score provides a harmonic mean of Precision and Recall on actual discrete predictions, directly capturing the business trade-off between false alarms and missed events.

#### 4. Superiority of Precision-Recall (PR) Curves over ROC on Imbalanced Data
The False Positive Rate (FPR) denominator in ROC curves is $FP / (FP + TN)$. When $TN$ is massive (e.g., thousands of negative samples), a large increase in false positives barely shifts the FPR, creating a deceptively optimistic ROC curve. In contrast, the Precision denominator is $TP / (TP + FP)$, which is completely uninflated by $TN$. Therefore, the Precision-Recall curve immediately exposes drops in precision, making it far more reliable for highly skewed datasets.

In [ ]:
# --- TASK 2: MULTI-MODEL BENCHMARKING (LR, RF, XGBOOST) ---
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

model_suite = {
    'Balanced Logistic Regression': lr_balanced,
    'Balanced Random Forest': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42).fit(X_train, y_train),
    'Weighted XGBoost': XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric='logloss', random_state=42).fit(X_train, y_train)
}

results_summary = []
fig, axes = plt.subplots(1, 2, figsize=(15, 6), dpi=300)
palette = ['#1f77b4', '#2ca02c', '#d62728']

for (name, model), col in zip(model_suite.items(), palette):
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    
    auc_score = roc_auc_score(y_test, y_prob)
    ap_score = average_precision_score(y_test, y_prob)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    results_summary.append({
        'Model Architecture': name,
        'ROC-AUC': round(auc_score, 4),
        'Avg Precision (PR-AUC)': round(ap_score, 4),
        'Recall (Default 0.5)': round(rec, 4),
        'F1-Score (Default 0.5)': round(f1, 4)
    })
    
    # ROC Curve
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC = {auc_score:.3f})", color=col, linewidth=2)
    
    # PR Curve
    prec, rec_curve, _ = precision_recall_curve(y_test, y_prob)
    axes[1].plot(rec_curve, prec, label=f"{name} (AP = {ap_score:.3f})", color=col, linewidth=2)

# Format ROC Plot
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='No-Skill (AUC = 0.50)')
axes[0].set_title("ROC Curves Comparison", fontsize=12, fontweight='bold')
axes[0].set_xlabel("False Positive Rate", fontsize=11)
axes[0].set_ylabel("True Positive Rate (Recall)", fontsize=11)
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].legend(loc='lower right', frameon=True)

# Format PR Plot
baseline_rate = (y_test == 1).sum() / len(y_test)
axes[1].plot([0, 1], [baseline_rate, baseline_rate], 'k--', alpha=0.5, label=f'Baseline Rate ({baseline_rate:.2f})')
axes[1].set_title("Precision-Recall Curves Comparison", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Recall", fontsize=11)
axes[1].set_ylabel("Precision", fontsize=11)
axes[1].grid(True, linestyle='--', alpha=0.5)
axes[1].legend(loc='upper right', frameon=True)

plt.tight_layout()
plt.savefig("roc_pr_curves.png", bbox_inches='tight', dpi=300)
plt.show()

print("\n=== Comprehensive Performance Benchmark ===")
print(pd.DataFrame(results_summary).to_markdown(index=False))

In [ ]:
# --- BONUS CHALLENGE: CUSTOM BUSINESS THRESHOLD OPTIMIZER ---
def optimize_threshold_for_recall(y_true, y_probabilities, min_recall_target=0.80):
    """
    Determines the decision threshold that satisfies a strict minimum recall constraint
    while maximizing precision on the minority class.
    """
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_probabilities)
    
    # Identify indices meeting the minimum recall target (excluding last recall value 0)
    qualifying_indices = np.where(recalls[:-1] >= min_recall_target)[0]
    
    if len(qualifying_indices) == 0:
        return 0.5, 0.0, 0.0
    
    # Select index with highest precision among qualifying thresholds
    best_qualifying_idx = qualifying_indices[np.argmax(precisions[qualifying_indices])]
    
    calibrated_threshold = thresholds[best_qualifying_idx]
    achieved_precision = precisions[best_qualifying_idx]
    achieved_recall = recalls[best_qualifying_idx]
    
    return float(calibrated_threshold), float(achieved_precision), float(achieved_recall)

# Apply to XGBoost probabilities
xgb_probs = model_suite['Weighted XGBoost'].predict_proba(X_test)[:, 1]
opt_thresh, opt_prec, opt_rec = optimize_threshold_for_recall(y_test, xgb_probs, min_recall_target=0.80)

print(f"=== XGBoost Threshold Calibration (Target Recall >= 80%) ===")
print(f"Calibrated Threshold : {opt_thresh:.4f}")
print(f"Achieved Recall      : {opt_rec:.4f} (>= 0.80 target satisfied)")
print(f"Achieved Precision   : {opt_prec:.4f}")

# Discrete predictions under new threshold
custom_preds = (xgb_probs >= opt_thresh).astype(int)
print(f"F1-Score at Calibrated Threshold: {f1_score(y_test, custom_preds):.4f}")
print("\nCalibrated Confusion Matrix:\n", confusion_matrix(y_test, custom_preds))